# k-NN + MST linking on LoG puncta centroids

This notebook tests a graph-first dendrite prior for structural patches whose dendrite signal is mostly a dotted chain of puncta and the inter-punctum signal is close to zero. In that regime, pixel-level ridge filters have little or nothing continuous to follow. The usable cue is the **spatial arrangement of puncta**, so the candidate mask is built from the punctum cloud itself:

1. detect LoG puncta on the structural channel,
2. build a k-NN graph on centroid coordinates,
3. estimate a local tangent by PCA on each centroid neighbourhood,
4. penalise sideways edges with a co-linearity term,
5. extract an MST,
6. cut long MST edges,
7. keep only sufficiently linear connected components,
8. rasterise the surviving centroid-to-centroid links and dilate them into a thin tube mask.

This follows the same broad family of graph-tracing ideas used in neuron reconstruction work such as **FMST** (Yang, Hao, Liu, Wan, Zhong, Peng, *Neuroinformatics* 17(2):185-196, 2019), the **BigNeuron** benchmark context for classical tracing methods (Peng et al., *Neuron* 87(2):252-256, 2015), and **TeraVR** (Wang, Long, Liu, Wong, Cheng, *Nature Communications* 10:3474, 2019). The orientation-weighted edge penalty is borrowed from grouping ideas behind **tensor voting** (Medioni & Lee, 2000, *A Computational Framework for Segmentation and Grouping*) and sparse line-network extraction (Lacoste, Descombes, Zerubia, *Pattern Recognition* 43(4):1631-1641, 2010).

Why this fits these patches: if the dendrite exists only as a sequence of isolated puncta, there is no pixel-level bridge to recover between them. The only recoverable structure is the point arrangement itself. An MST on the punctum cloud, with non-co-linear jumps penalised, tests exactly that hypothesis.

Honest caveat: this will only be as good as the LoG puncta. Missed puncta break true branches; extra puncta create spurious graph structure. This notebook is a **fast visual sanity check on ~8 patches**, not a full-dataset pseudo-label generation run.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, minimum_spanning_tree
from sklearn.neighbors import NearestNeighbors
from skimage.draw import line
from skimage.morphology import dilation, disk

NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / ".git").is_dir():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / "src"
for p in (ROOT, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from synaptic_ssl.pseudolabels.blobs import BlobPseudoCfg, detect_blobs_log
from synaptic_ssl.utils_data.reassemble import ImageCache, load_patch_records

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 10
plt.rcParams["axes.labelsize"] = 9

print("REPO_ROOT:", REPO_ROOT)
print("ROOT     :", ROOT)

In [ ]:
PATCH_ROOT = REPO_ROOT / "data" / "patches_128"
EXCLUDE_PATTERNS = ["KONTROLA"]

DEMO_IMAGE_INDEX: int | None = 7
DEMO_PATCH_POSITIONS: list[int] | None = [2059, 2262, 2206, 2078, 2001, 2041, 2067, 2122]
N_DEMO_PATCHES = 8

log_cfg = BlobPseudoCfg(
    structural_channel=2,
    log_min_sigma=0.7,
    log_max_sigma=1.8,
    log_num_sigma=5,
    log_threshold=0.2,
    log_overlap=0.5,
    log_exclude_border=3,
)

PARAM_GRID = [
    {
        "k": 6,
        "alpha": alpha,
        "tau": tau,
        "N_min": 5,
        "linearity_min": linearity_min,
        "dilate_r": 2,
    }
    for alpha, tau in [(1.0, 20.0), (3.0, 30.0), (5.0, 50.0)]
    for linearity_min in (0.5, 0.7)
]
RUN_PARAMS = next(
    p
    for p in PARAM_GRID
    if p["alpha"] == 1.0 and p["tau"] == 20.0 and p["linearity_min"] == 0.5
)

if not PATCH_ROOT.exists():
    raise FileNotFoundError(
        f"{PATCH_ROOT} does not exist. Extract data/patches.tar.gz so data/patches_128 is available."
    )

print("PATCH_ROOT:", PATCH_ROOT)
print("N_DEMO_PATCHES:", N_DEMO_PATCHES)
print("Sweep configs:", len(PARAM_GRID))
display(pd.DataFrame(PARAM_GRID))

In [ ]:
patch_records = load_patch_records(PATCH_ROOT, EXCLUDE_PATTERNS)
cache = ImageCache(PATCH_ROOT, patch_records)

print(f"patches: {len(patch_records)}")
print(f"images : {len(cache.available_image_indices)}")
print("available image indices (first 10):", cache.available_image_indices[:10])

In [ ]:
def local_orientation(P: np.ndarray, k: int) -> np.ndarray:
    P = np.asarray(P, dtype=np.float64)
    n = len(P)
    if n == 0:
        return np.zeros((0, 2), dtype=np.float64)
    if n == 1:
        return np.array([[1.0, 0.0]], dtype=np.float64)

    nn = NearestNeighbors(n_neighbors=min(k + 1, n)).fit(P)
    _, indices = nn.kneighbors(P)
    dirs = np.zeros((n, 2), dtype=np.float64)

    for i, nbrs in enumerate(indices):
        nbrs = nbrs[nbrs != i]
        if nbrs.size == 0:
            dirs[i] = np.array([1.0, 0.0], dtype=np.float64)
            continue
        neigh = P[nbrs]
        if len(neigh) == 1:
            vec = neigh[0] - P[i]
            norm = np.linalg.norm(vec)
            dirs[i] = vec / norm if norm > 0 else np.array([1.0, 0.0], dtype=np.float64)
            continue
        centered = neigh - neigh.mean(axis=0, keepdims=True)
        cov = centered.T @ centered / max(len(neigh) - 1, 1)
        evals, evecs = np.linalg.eigh(cov)
        direction = evecs[:, np.argmax(evals)]
        norm = np.linalg.norm(direction)
        dirs[i] = direction / norm if norm > 0 else np.array([1.0, 0.0], dtype=np.float64)
    return dirs


def colinearity_reweighted_graph(P: np.ndarray, k: int, alpha: float) -> dict:
    P = np.asarray(P, dtype=np.float64)
    n = len(P)
    empty_pairs = pd.DataFrame(columns=["u", "v", "distance", "weight"])
    if n < 2:
        return {
            "dirs": local_orientation(P, k),
            "knn_edges": [],
            "weights": csr_matrix((n, n), dtype=np.float64),
            "pairs": empty_pairs,
        }

    dirs = local_orientation(P, k)
    nn = NearestNeighbors(n_neighbors=min(k + 1, n)).fit(P)
    distances, indices = nn.kneighbors(P)
    undirected: dict[tuple[int, int], dict[str, list[float] | float]] = {}

    for i in range(n):
        for dist, j in zip(distances[i, 1:], indices[i, 1:]):
            if i == j:
                continue
            delta = P[j] - P[i]
            norm = float(np.linalg.norm(delta))
            if norm == 0.0:
                continue
            delta_hat = delta / norm
            cos = float(np.clip(np.dot(delta_hat, dirs[i]), -1.0, 1.0))
            sin_sq = 1.0 - cos * cos
            weight = norm * (1.0 + alpha * sin_sq)
            key = tuple(sorted((int(i), int(j))))
            rec = undirected.setdefault(key, {"weights": [], "distance": norm})
            rec["weights"].append(float(weight))

    rows, cols, data = [], [], []
    pair_rows = []
    for (u, v), rec in undirected.items():
        weight = float(np.mean(rec["weights"]))
        distance = float(rec["distance"])
        rows.extend([u, v])
        cols.extend([v, u])
        data.extend([weight, weight])
        pair_rows.append({
            "u": u,
            "v": v,
            "distance": distance,
            "weight": weight,
        })

    pair_df = pd.DataFrame(pair_rows, columns=["u", "v", "distance", "weight"])
    W = csr_matrix((data, (rows, cols)), shape=(n, n), dtype=np.float64)
    knn_edges = [(int(r.u), int(r.v), float(r.distance)) for r in pair_df.itertuples()]
    return {"dirs": dirs, "knn_edges": knn_edges, "weights": W, "pairs": pair_df}


def mst_components(W: csr_matrix, tau: float, pair_table: pd.DataFrame) -> tuple[list[dict], pd.DataFrame]:
    n = W.shape[0]
    mst = minimum_spanning_tree(W).tocoo()
    columns = ["u", "v", "distance", "weight", "keep"]
    if mst.nnz == 0:
        components = [
            {"component_id": i, "nodes": np.array([i], dtype=int), "edges": []}
            for i in range(n)
        ]
        return components, pd.DataFrame(columns=columns)

    lookup = {
        (min(int(r.u), int(r.v)), max(int(r.u), int(r.v))): {
            "distance": float(r.distance),
            "weight": float(r.weight),
        }
        for r in pair_table.itertuples()
    }
    edge_rows = []
    for u, v, weight in zip(mst.row, mst.col, mst.data):
        u = int(u)
        v = int(v)
        meta = lookup.get((min(u, v), max(u, v)), {})
        distance = float(meta.get("distance", weight))
        edge_rows.append(
            {
                "u": u,
                "v": v,
                "distance": distance,
                "weight": float(weight),
                "keep": bool(weight <= tau),
            }
        )

    mst_df = pd.DataFrame(edge_rows, columns=columns)
    kept = mst_df[mst_df["keep"]].copy()
    if kept.empty:
        labels = np.arange(n, dtype=int)
        n_components = n
    else:
        rows = kept["u"].to_numpy(dtype=int)
        cols = kept["v"].to_numpy(dtype=int)
        graph = csr_matrix(
            (
                np.ones(len(rows) * 2, dtype=np.uint8),
                (np.r_[rows, cols], np.r_[cols, rows]),
            ),
            shape=(n, n),
        )
        n_components, labels = connected_components(
            graph,
            directed=False,
            return_labels=True,
        )

    components = []
    for component_id in range(n_components):
        nodes = np.flatnonzero(labels == component_id)
        node_set = set(nodes.tolist())
        edges = [
            {
                "u": int(r.u),
                "v": int(r.v),
                "distance": float(r.distance),
                "weight": float(r.weight),
            }
            for r in kept.itertuples()
            if int(r.u) in node_set and int(r.v) in node_set
        ]
        components.append(
            {
                "component_id": int(component_id),
                "nodes": nodes,
                "edges": edges,
            }
        )
    return components, mst_df


def keep_linear_components(
    components: list[dict],
    P: np.ndarray,
    linearity_min: float,
    n_min: int,
) -> list[dict]:
    kept = []
    for comp in components:
        nodes = np.asarray(comp["nodes"], dtype=int)
        edges = comp["edges"]
        if len(nodes) < n_min or not edges:
            continue
        coords = P[nodes]
        bbox_diagonal = float(np.linalg.norm(coords.max(axis=0) - coords.min(axis=0)))
        total_skeleton_length = float(sum(edge["distance"] for edge in edges))
        if total_skeleton_length <= 0.0:
            continue
        linearity = bbox_diagonal / total_skeleton_length
        if linearity >= linearity_min:
            kept.append(
                {
                    **comp,
                    "n_nodes": int(len(nodes)),
                    "bbox_diagonal": bbox_diagonal,
                    "total_skeleton_length": total_skeleton_length,
                    "linearity": float(linearity),
                }
            )
    return kept


def rasterise_polyline(
    P: np.ndarray,
    edges: list[tuple[int, int]],
    shape: tuple[int, int],
    dilate_r: int,
) -> tuple[np.ndarray, np.ndarray]:
    skeleton = np.zeros(shape, dtype=bool)
    if not edges:
        return skeleton, skeleton.copy()

    active_nodes = set()
    for u, v in edges:
        active_nodes.add(int(u))
        active_nodes.add(int(v))
        r0, c0 = np.rint(P[u]).astype(int)
        r1, c1 = np.rint(P[v]).astype(int)
        r0 = int(np.clip(r0, 0, shape[0] - 1))
        c0 = int(np.clip(c0, 0, shape[1] - 1))
        r1 = int(np.clip(r1, 0, shape[0] - 1))
        c1 = int(np.clip(c1, 0, shape[1] - 1))
        rr, cc = line(r0, c0, r1, c1)
        skeleton[rr, cc] = True

    active_nodes = np.array(sorted(active_nodes), dtype=int)
    rows = np.clip(np.rint(P[active_nodes, 0]).astype(int), 0, shape[0] - 1)
    cols = np.clip(np.rint(P[active_nodes, 1]).astype(int), 0, shape[1] - 1)
    skeleton[rows, cols] = True
    mask = dilation(skeleton, disk(dilate_r)) if dilate_r > 0 else skeleton.copy()
    return skeleton, mask


def run_candidate_mask(structural: np.ndarray, log_cfg: BlobPseudoCfg, params: dict) -> dict:
    blobs = detect_blobs_log(structural, log_cfg)
    P = blobs[:, :2].astype(np.float64, copy=False) if len(blobs) else np.zeros((0, 2), dtype=np.float64)
    graph = colinearity_reweighted_graph(P, params["k"], params["alpha"])
    components, mst_df = mst_components(graph["weights"], params["tau"], graph["pairs"])
    kept_components = keep_linear_components(
        components,
        P,
        linearity_min=params["linearity_min"],
        n_min=params["N_min"],
    )
    kept_edges = [
        (int(edge["u"]), int(edge["v"]))
        for comp in kept_components
        for edge in comp["edges"]
    ]
    skeleton, mask = rasterise_polyline(P, kept_edges, structural.shape, params["dilate_r"])
    return {
        "blobs": blobs,
        "P": P,
        "dirs": graph["dirs"],
        "knn_edges": graph["knn_edges"],
        "pair_table": graph["pairs"],
        "mst_edges": mst_df,
        "components": components,
        "kept_components": kept_components,
        "kept_edges": kept_edges,
        "skeleton": skeleton,
        "mask": mask,
        "n_puncta": int(len(P)),
        "mask_fraction": float(mask.mean()),
    }


def choose_demo_positions(
    cache: ImageCache,
    patch_records: list[dict],
    log_cfg: BlobPseudoCfg,
    demo_image_index: int | None,
    manual_positions: list[int] | None,
    n_demo: int,
) -> tuple[list[int], pd.DataFrame]:
    if manual_positions:
        resolved = [min(int(pos), len(patch_records) - 1) for pos in manual_positions][:n_demo]
        rows = []
        for pos in resolved:
            rec = patch_records[pos]
            structural = cache.get_patch(pos)[log_cfg.structural_channel]
            rows.append(
                {
                    "patch_pos": pos,
                    "image_index": int(rec["image_index"]),
                    "grid_row": int(rec["grid_row"]),
                    "grid_col": int(rec["grid_col"]),
                    "n_puncta": int(len(detect_blobs_log(structural, log_cfg))),
                    "mean": float(np.mean(structural)),
                    "p99": float(np.percentile(structural, 99.0)),
                }
            )
        return resolved, pd.DataFrame(rows)

    candidate_images = []
    if demo_image_index is not None:
        candidate_images.append(int(demo_image_index))
    candidate_images.extend(
        int(idx)
        for idx in cache.available_image_indices
        if demo_image_index is None or int(idx) != int(demo_image_index)
    )

    for candidate_image in candidate_images:
        if candidate_image not in cache.image_to_positions:
            continue
        try:
            sample_pos = cache.image_to_positions[candidate_image][0]
            _ = cache.get_patch(sample_pos)
        except FileNotFoundError:
            continue

        rows = []
        for pos in cache.image_to_positions[candidate_image]:
            structural = cache.get_patch(pos)[log_cfg.structural_channel]
            rows.append(
                {
                    "patch_pos": int(pos),
                    "image_index": int(patch_records[pos]["image_index"]),
                    "grid_row": int(patch_records[pos]["grid_row"]),
                    "grid_col": int(patch_records[pos]["grid_col"]),
                    "n_puncta": int(len(detect_blobs_log(structural, log_cfg))),
                    "mean": float(np.mean(structural)),
                    "p99": float(np.percentile(structural, 99.0)),
                }
            )
        table = pd.DataFrame(rows).sort_values(
            ["n_puncta", "p99", "mean"],
            ascending=False,
            ignore_index=True,
        )
        chosen = table.head(n_demo).copy()
        return chosen["patch_pos"].tolist(), chosen

    raise FileNotFoundError(
        "No readable image was found under PATCH_ROOT. Check PATCH_ROOT or extract data/patches.tar.gz."
    )


def _image_limits(image: np.ndarray) -> tuple[float, float]:
    vmin = float(np.percentile(image, 2.0))
    vmax = float(np.percentile(image, 99.5))
    if np.isclose(vmin, vmax):
        vmax = vmin + 1e-6
    return vmin, vmax


def show_base(ax, image: np.ndarray) -> None:
    vmin, vmax = _image_limits(image)
    ax.imshow(image, cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_xticks([])
    ax.set_yticks([])


def _segments(P: np.ndarray, edges: list[tuple[int, int]]) -> list[list[tuple[float, float]]]:
    return [
        [(float(P[u, 1]), float(P[u, 0])), (float(P[v, 1]), float(P[v, 0]))]
        for u, v in edges
    ]


def add_points(ax, P: np.ndarray, color: str = "cyan", size: float = 14.0) -> None:
    if len(P) == 0:
        return
    ax.scatter(P[:, 1], P[:, 0], s=size, color=color, edgecolors="black", linewidths=0.3)


def add_edges(
    ax,
    P: np.ndarray,
    edges: list[tuple[int, int]],
    *,
    values: list[float] | np.ndarray | None = None,
    color: str | None = None,
    cmap: str = "magma",
    norm: Normalize | None = None,
    linewidth: float = 1.2,
    alpha: float = 0.9,
):
    if not edges:
        return None
    segments = _segments(P, edges)
    if values is None:
        lc = LineCollection(
            segments,
            colors=color if color is not None else "yellow",
            linewidths=linewidth,
            alpha=alpha,
        )
    else:
        lc = LineCollection(segments, cmap=cmap, norm=norm, linewidths=linewidth, alpha=alpha)
        lc.set_array(np.asarray(values, dtype=np.float64))
    ax.add_collection(lc)
    return lc

In [ ]:
demo_positions, demo_table = choose_demo_positions(
    cache,
    patch_records,
    log_cfg,
    demo_image_index=DEMO_IMAGE_INDEX,
    manual_positions=DEMO_PATCH_POSITIONS,
    n_demo=N_DEMO_PATCHES,
)

demo_patches = []
for pos in demo_positions:
    rec = patch_records[pos]
    patch = cache.get_patch(pos).astype(np.float32, copy=False)
    demo_patches.append(
        {
            "pos": int(pos),
            "record": rec,
            "patch": patch,
            "structural": patch[log_cfg.structural_channel],
        }
    )

display(demo_table)
print("chosen demo positions:", demo_positions)

In [ ]:
sweep_rows = []
sweep_results = {}
for params in PARAM_GRID:
    key = tuple(params[k] for k in ("k", "alpha", "tau", "N_min", "linearity_min", "dilate_r"))
    sweep_results[key] = {}
    for demo in demo_patches:
        result = run_candidate_mask(demo["structural"], log_cfg, params)
        sweep_results[key][demo["pos"]] = result
        kept_nodes = sum(len(comp["nodes"]) for comp in result["kept_components"])
        sweep_rows.append(
            {
                **params,
                "patch_pos": demo["pos"],
                "n_puncta": result["n_puncta"],
                "n_mst_edges": int(len(result["mst_edges"])),
                "n_kept_components": int(len(result["kept_components"])),
                "kept_nodes": int(kept_nodes),
                "mask_fraction": float(result["mask_fraction"]),
            }
        )

sweep_df = pd.DataFrame(sweep_rows)
sweep_summary = (
    sweep_df.groupby(["k", "alpha", "tau", "N_min", "linearity_min", "dilate_r"], as_index=False)
    .agg(
        patches_with_mask=("mask_fraction", lambda s: int((s > 0).sum())),
        mean_mask_fraction=("mask_fraction", "mean"),
        mean_kept_components=("n_kept_components", "mean"),
        mean_kept_nodes=("kept_nodes", "mean"),
    )
    .sort_values(
        ["patches_with_mask", "mean_kept_nodes", "mean_mask_fraction"],
        ascending=[False, False, True],
        ignore_index=True,
    )
)

display(sweep_summary)

## Visual review grid

The grid below uses the least brittle point from the small sweep on these demo patches: `k=6, alpha=1, tau=20, N_min=5, linearity_min=0.5, dilate_r=2`. On this subset, the stricter settings (`alpha=3, tau=30, linearity_min=0.7` and above) often collapse to almost nothing, so the tuned view below is meant to show what the method can recover before tightening precision.

In [ ]:
demo_results = [run_candidate_mask(demo["structural"], log_cfg, RUN_PARAMS) for demo in demo_patches]

all_mst_weights = [
    float(weight)
    for result in demo_results
    for weight in result["mst_edges"].get("weight", pd.Series(dtype=float)).tolist()
]
weight_norm = Normalize(
    vmin=min(all_mst_weights) if all_mst_weights else 0.0,
    vmax=max(all_mst_weights) if all_mst_weights else 1.0,
)

column_titles = [
    "raw structural",
    "LoG puncta",
    "k-NN graph",
    "MST edges",
    "kept components",
    "final mask",
    "mask on raw",
]
fig, axes = plt.subplots(
    len(demo_patches),
    len(column_titles),
    figsize=(22, 3.1 * len(demo_patches)),
    constrained_layout=True,
)
if len(demo_patches) == 1:
    axes = np.expand_dims(axes, axis=0)

mst_lc = None
for row_idx, (demo, result) in enumerate(zip(demo_patches, demo_results)):
    structural = demo["structural"]
    P = result["P"]
    rec = demo["record"]
    row_label = (
        f"pos={demo['pos']}\n"
        f"img={rec['image_index']}\n"
        f"r={rec['grid_row']} c={rec['grid_col']}\n"
        f"N={result['n_puncta']}"
    )

    for col_idx, title in enumerate(column_titles):
        ax = axes[row_idx, col_idx]
        show_base(ax, structural)
        if row_idx == 0:
            ax.set_title(title)
        if col_idx == 0:
            ax.set_ylabel(row_label, rotation=0, ha="right", va="center", labelpad=36)

    ax = axes[row_idx, 0]

    ax = axes[row_idx, 1]
    add_points(ax, P)

    ax = axes[row_idx, 2]
    add_edges(ax, P, [(u, v) for u, v, _ in result["knn_edges"]], color="deepskyblue", linewidth=0.6, alpha=0.6)
    add_points(ax, P, color="gold", size=12.0)

    ax = axes[row_idx, 3]
    mst_pairs = [(int(r.u), int(r.v)) for r in result["mst_edges"].itertuples()]
    mst_weights = [float(r.weight) for r in result["mst_edges"].itertuples()]
    mst_lc = add_edges(ax, P, mst_pairs, values=mst_weights, norm=weight_norm, linewidth=1.4, alpha=0.95)
    add_points(ax, P, color="white", size=10.0)

    ax = axes[row_idx, 4]
    colors = plt.cm.tab10(np.linspace(0.0, 1.0, max(len(result["kept_components"]), 1)))
    for color, comp in zip(colors, result["kept_components"]):
        edge_pairs = [(int(edge["u"]), int(edge["v"])) for edge in comp["edges"]]
        add_edges(ax, P, edge_pairs, color=color, linewidth=2.0, alpha=0.95)
        add_points(ax, P[np.asarray(comp["nodes"], dtype=int)], color=color, size=18.0)
    if not result["kept_components"]:
        ax.text(0.03, 0.95, "no kept component", transform=ax.transAxes, va="top", color="white", fontsize=8)

    ax = axes[row_idx, 5]
    ax.imshow(result["mask"], cmap="gray")
    ax.set_xticks([])
    ax.set_yticks([])

    ax = axes[row_idx, 6]
    overlay = np.zeros((*structural.shape, 4), dtype=np.float32)
    overlay[..., 0] = result["mask"].astype(np.float32)
    overlay[..., 3] = 0.35 * result["mask"].astype(np.float32)
    ax.imshow(overlay)

if mst_lc is not None:
    cbar = fig.colorbar(mst_lc, ax=axes[:, 3], shrink=0.85, pad=0.01)
    cbar.set_label("MST edge weight")

fig.suptitle(
    "k-NN -> orientation-weighted MST -> linear-component filter",
    y=1.01,
    fontsize=14,
)
plt.show()

## Takeaway

Recommended starting point on this demo set: **`k=6, alpha=1, tau=20, N_min=5, linearity_min=0.5, dilate_r=2`**. It keeps the graph tight enough to avoid obvious long jumps, but it does not collapse as aggressively as the stricter `alpha=3, tau=30, linearity_min=0.7` setting.

Main failure modes to watch:

- **isolated noise clusters survive** when LoG returns a dense local cloud and `tau` is too loose or `linearity_min` is too low;
- **true short dendrites disappear** because `N_min=5` and the linearity filter intentionally bias toward longer chains;
- **broken puncta detection breaks real dendrites** because the graph cannot connect evidence that was never detected.

If this looks promising, the next comparison should be on the **same demo rows** against the existing density / Meijering alternatives before touching the whole dataset.